# Q3: Return Pattern Analysis

### Business Question
Are cancellations or returns concentrated in specific months, products, or customer segments?

### Objective
To quantify return rates over time and assess whether returns pose structural revenue risk.

In [78]:
import pandas as pd
df = pd.read_csv('../data/processed/retail_with_returns.csv')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,TotalSales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1061164 entries, 0 to 1061163
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1061164 non-null  object 
 1   StockCode    1061164 non-null  object 
 2   Description  1061164 non-null  object 
 3   Quantity     1061164 non-null  int64  
 4   InvoiceDate  1061164 non-null  object 
 5   Price        1061164 non-null  float64
 6   Customer_ID  824293 non-null   float64
 7   Country      1061164 non-null  object 
 8   TotalSales   1061164 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 72.9+ MB


In [80]:
## Create a new column 'Is_Return' to identify returns based on the 'Invoice' column
df['Is_Return'] = df['Invoice'].astype(str).str.startswith('C')
df['Invoice'].astype(str).str.startswith('C').sum()


19494

In [81]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')


In [82]:
## Calculate total returns and gross sales by month
monthly_gross = df[~df['Is_Return']].groupby('YearMonth')['TotalSales'].sum()
print(monthly_gross)

YearMonth
2009-12     825685.760
2010-01     652708.502
2010-02     553339.736
2010-03     833570.131
2010-04     681528.992
2010-05     659858.860
2010-06     752270.140
2010-07     650712.940
2010-08     697274.910
2010-09     924333.011
2010-10    1165483.910
2010-11    1470272.482
2010-12    1262598.790
2011-01     691364.560
2011-02     523631.890
2011-03     717639.360
2011-04     537808.621
2011-05     770536.020
2011-06     761739.900
2011-07     719221.191
2011-08     759138.380
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638810.680
Freq: M, Name: TotalSales, dtype: float64


In [83]:
monthly_returns = df[df['Is_Return']].groupby('YearMonth')['TotalSales'].sum()
print(monthly_returns)

YearMonth
2009-12    -25838.65
2010-01    -28675.61
2010-02    -20248.31
2010-03    -67721.37
2010-04    -37354.20
2010-05    -44536.03
2010-06    -72483.53
2010-07    -31444.79
2010-08    -40498.57
2010-09    -70682.58
2010-10    -81389.69
2010-11    -47617.84
2010-12   -136153.32
2011-01   -131364.30
2011-02    -25569.24
2011-03    -34372.28
2011-04    -44601.50
2011-05    -47202.51
2011-06    -70616.78
2011-07    -37921.08
2011-08    -54333.75
2011-09    -38902.55
2011-10    -84274.63
2011-11    -47740.08
2011-12   -205124.67
Freq: M, Name: TotalSales, dtype: float64


In [84]:

monthly_return_rate = abs(monthly_returns) / monthly_gross * 100
print(monthly_return_rate)

YearMonth
2009-12     3.129356
2010-01     4.393326
2010-02     3.659291
2010-03     8.124256
2010-04     5.480941
2010-05     6.749327
2010-06     9.635306
2010-07     4.832360
2010-08     5.808121
2010-09     7.646874
2010-10     6.983339
2010-11     3.238709
2010-12    10.783578
2011-01    19.000728
2011-02     4.883056
2011-03     4.789631
2011-04     8.293192
2011-05     6.125932
2011-06     9.270458
2011-07     5.272520
2011-08     7.157292
2011-09     3.674940
2011-10     7.296636
2011-11     3.162650
2011-12    32.110401
Freq: M, Name: TotalSales, dtype: float64


## Return rate result
Generally stable between 3–8%

Spike in Jan 2011 (post-holiday effect)

Dec 2011 inflated due to incomplete month

In [85]:
## Analyze product-level return rates
df_products = df_products[df_products['StockCode'].str.len() == 5]  # Filter to keep only products (exclude services)
product_gross = df_products[~df_products['Is_Return']].groupby('StockCode')['TotalSales'].sum()
print(product_gross.sort_values(ascending=False).head(10))

StockCode
22423    344563.25
23843    168469.60
47566    149187.05
84879    132187.92
22086    123141.54
79321     85489.91
23166     81700.92
22197     80920.64
22386     77111.91
84347     74448.92
Name: TotalSales, dtype: float64


In [86]:
product_returns = df_products[df_products['Is_Return']].groupby('StockCode')['TotalSales'].sum()
product_return_rate = abs(product_returns) / product_gross * 100
print(product_return_rate.sort_values(ascending=False).head(10))

StockCode
79301    777.966102
85043    300.000000
20879    266.666667
85069    200.000000
37451    171.717172
85083    120.000000
20885    100.000000
23843    100.000000
21525    100.000000
20822    100.000000
Name: TotalSales, dtype: float64


In [87]:
## Filter to products with significant sales to avoid skewed return rates from low-volume items
Min_sales_threshold = 5000
valid_products = product_gross[product_gross >= Min_sales_threshold].index
product_return_rate_filtered = product_return_rate.loc[valid_products]
print(product_return_rate_filtered.sort_values(ascending=False).head(10))

StockCode
23843    100.000000
23166     94.833253
23113     81.844764
85220     53.027636
22341     32.436622
21108     25.550587
22656     25.495156
22670     23.978779
21735     22.042658
22085     21.740956
Name: TotalSales, dtype: float64


In [88]:
df_products[df_products['StockCode'] == '23843']

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,TotalSales,Is_Return,YearMonth
1059675,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.6,False,2011-12
1059676,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,-168469.6,True,2011-12


In [89]:
df_products[df_products['StockCode'] == '23166']

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,TotalSales,Is_Return,YearMonth
583060,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60,False,2011-01
583065,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,-77183.60,True,2011-01
707411,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,2011-05-12 10:10:00,1.04,14646.0,Netherlands,99.84,False,2011-05
707837,552953,23166,MEDIUM CERAMIC TOP STORAGE JAR,4,2011-05-12 12:11:00,1.25,16745.0,United Kingdom,5.00,False,2011-05
708357,553005,23166,MEDIUM CERAMIC TOP STORAGE JAR,5,2011-05-12 16:29:00,1.25,14651.0,United Kingdom,6.25,False,2011-05
...,...,...,...,...,...,...,...,...,...,...,...
1053019,581108,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,2011-12-07 12:16:00,1.25,15984.0,United Kingdom,2.50,False,2011-12
1055508,581219,23166,MEDIUM CERAMIC TOP STORAGE JAR,1,2011-12-08 09:28:00,2.46,NaN,United Kingdom,2.46,False,2011-12
1059030,581439,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,2011-12-08 16:30:00,2.46,NaN,United Kingdom,4.92,False,2011-12
1059555,581476,23166,MEDIUM CERAMIC TOP STORAGE JAR,48,2011-12-09 08:48:00,1.04,12433.0,Norway,49.92,False,2011-12


In [90]:
df_products[df_products['StockCode'] == '85220']

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,TotalSales,Is_Return,YearMonth
3307,489676,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,240,2009-12-02 09:49:00,1.45,13777.0,United Kingdom,348.00,False,2009-12
25393,C491594,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,-9,2009-12-11 13:13:00,1.65,12921.0,United Kingdom,-14.85,True,2009-12
32845,492092,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,12,2009-12-15 14:03:00,1.65,14156.0,EIRE,19.80,False,2009-12
56136,494473,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,24,2010-01-14 14:53:00,1.65,17508.0,Greece,39.60,False,2010-01
90028,497946,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,2504,2010-02-15 11:57:00,1.45,13902.0,Denmark,3630.80,False,2010-02
92798,C498151,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,-2504,2010-02-17 10:37:00,1.45,13902.0,Denmark,-3630.80,True,2010-02
92799,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,2010-02-17 10:51:00,0.30,13902.0,Denmark,2836.80,False,2010-02


## Product Return
Several SKUs show >50% return rates

Drill-down analysis reveals bulk wholesale reversals

No evidence of systemic product defect

In [91]:
df_goods = df[df['StockCode'].str.len() == 5]
customer_returns = df_goods[df_goods['Is_Return']].groupby('Customer_ID')['TotalSales'].sum()
customer_gross = df_goods[~df_goods['Is_Return']].groupby('Customer_ID')['TotalSales'].sum()
customer_return_rate = abs(customer_returns) / customer_gross * 100
customer_return_rate_filtered = customer_return_rate[customer_return_rate > 0]
print(customer_return_rate_filtered.sort_values(ascending=False).head(10))


Customer_ID
15935.0    255.572010
13915.0    223.728814
13091.0    203.275500
14213.0    200.000000
16252.0    200.000000
12768.0    100.000000
12607.0    100.000000
18274.0    100.000000
16886.0    100.000000
16878.0    100.000000
Name: TotalSales, dtype: float64


In [92]:
min_customer_sales_threshold = 10000
valid_customers = customer_gross[customer_gross >= min_customer_sales_threshold].index
customer_return_rate_filtered = customer_return_rate.loc[valid_customers]
print(customer_return_rate_filtered.sort_values(ascending=False).head(10))

Customer_ID
16446.0    99.998279
12346.0    99.813753
12454.0    73.835613
14277.0    71.695998
15749.0    51.016042
14028.0    34.728033
15482.0    33.900630
14607.0    26.916567
12931.0    22.524060
16754.0    19.208353
Name: TotalSales, dtype: float64


In [93]:
customer_return_rate_filtered.describe()

count    202.000000
mean       4.773862
std       13.176036
min        0.007950
25%        0.732058
50%        1.453832
75%        2.990730
max       99.998279
Name: TotalSales, dtype: float64

In [96]:
df[df['Customer_ID'] == 16446]


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,TotalSales,Is_Return,YearMonth
368842,C525275,TEST001,This is a test product.,-2,2010-10-04 16:38:00,4.50,16446.0,United Kingdom,-9.00,True,2010-10
714906,553573,22980,PANTRY SCRUBBING BRUSH,1,2011-05-18 09:52:00,1.65,16446.0,United Kingdom,1.65,False,2011-05
714907,553573,22982,PANTRY PASTRY BRUSH,1,2011-05-18 09:52:00,1.25,16446.0,United Kingdom,1.25,False,2011-05
1059675,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60,False,2011-12
1059676,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,-168469.60,True,2011-12


In [97]:
df[df['Customer_ID'] == 12346]


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,TotalSales,Is_Return,YearMonth
27923,491725,TEST001,This is a test product.,10,2009-12-14 08:34:00,4.50,12346.0,United Kingdom,45.00,False,2009-12
28179,491742,TEST001,This is a test product.,5,2009-12-14 11:00:00,4.50,12346.0,United Kingdom,22.50,False,2009-12
28182,491744,TEST001,This is a test product.,5,2009-12-14 11:02:00,4.50,12346.0,United Kingdom,22.50,False,2009-12
39295,492718,TEST001,This is a test product.,5,2009-12-18 10:47:00,4.50,12346.0,United Kingdom,22.50,False,2009-12
39307,492722,TEST002,This is a test product.,1,2009-12-18 10:55:00,1.00,12346.0,United Kingdom,1.00,False,2009-12
44972,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346.0,United Kingdom,22.50,False,2010-01
44974,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346.0,United Kingdom,22.50,False,2010-01
55773,494450,TEST001,This is a test product.,5,2010-01-14 13:50:00,4.50,12346.0,United Kingdom,22.50,False,2010-01
65696,495295,TEST001,This is a test product.,5,2010-01-22 13:30:00,4.50,12346.0,United Kingdom,22.50,False,2010-01
70626,C495800,ADJUST,Adjustment by john on 26/01/2010 17,-1,2010-01-26 17:27:00,103.50,12346.0,United Kingdom,-103.50,True,2010-01


## Customer-Level Insights
After excluding non-product adjustment codes, the median customer return rate is 1.45%, with 75% of customers below 3%. 

Although a few wholesale accounts exhibit extremely high return rates (approaching 100%), transaction-level inspection reveals that these cases are driven by large order reversals and commercial adjustments rather than systematic product defects.

This suggests that return risk is concentrated among a small number of high-volume B2B customers, reflecting wholesale operational behavior rather than widespread quality issues.